In [3]:
import os
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import datetime

class ResearchArtifactManager:

    def __init__(self):

        self.base_path = r"D:\Med_data"
        self.dataset_path = os.path.join(self.base_path, "healthcare_dataset.csv")

        self.diagrams_path = os.path.join(self.base_path, "Diagrams")
        self.xml_path = os.path.join(self.base_path, "XML")
        self.report_path = os.path.join(self.base_path, "Reports")

        self.stages = ["Stage1","Stage2","Stage3","Stage4","Stage5","Stage6"]

        self.figure_counter = {stage:0 for stage in self.stages}

        print("\n=== Research Artifact Manager Initialized ===")

    # --------------------------------------------------
    # Create folder structure
    # --------------------------------------------------

    def create_directories(self):

        print("\nCreating folder structure...")

        os.makedirs(self.diagrams_path, exist_ok=True)
        os.makedirs(self.xml_path, exist_ok=True)
        os.makedirs(self.report_path, exist_ok=True)

        for stage in self.stages:

            os.makedirs(os.path.join(self.diagrams_path, stage), exist_ok=True)
            os.makedirs(os.path.join(self.xml_path, stage), exist_ok=True)

        print("Folder structure created successfully.")

    # --------------------------------------------------
    # Load dataset
    # --------------------------------------------------

    def load_dataset(self):

        print("\nLoading dataset...")

        if not os.path.exists(self.dataset_path):
            raise FileNotFoundError(f"Dataset not found at {self.dataset_path}")

        df = pd.read_csv(self.dataset_path)

        print("Dataset loaded successfully.")
        print("Rows:", df.shape[0])
        print("Columns:", df.shape[1])

        return df

    # --------------------------------------------------
    # Generate dataset profile XML
    # --------------------------------------------------

    def generate_dataset_profile_xml(self, df):

        print("\nGenerating dataset profile XML...")

        root = ET.Element("DatasetProfile")

        meta = ET.SubElement(root, "Metadata")
        ET.SubElement(meta, "Rows").text = str(df.shape[0])
        ET.SubElement(meta, "Columns").text = str(df.shape[1])
        ET.SubElement(meta, "GeneratedOn").text = str(datetime.now())

        columns = ET.SubElement(root, "Columns")

        for col in df.columns:

            col_node = ET.SubElement(columns, "Column")

            ET.SubElement(col_node, "Name").text = col
            ET.SubElement(col_node, "Datatype").text = str(df[col].dtype)
            ET.SubElement(col_node, "MissingValues").text = str(df[col].isnull().sum())
            ET.SubElement(col_node, "UniqueValues").text = str(df[col].nunique())

        tree = ET.ElementTree(root)

        output_path = os.path.join(self.xml_path,"Stage1","dataset_profile.xml")
        tree.write(output_path)

        print("Dataset profile XML saved:", output_path)

    # --------------------------------------------------
    # Generate descriptive statistics XML
    # --------------------------------------------------

    def generate_descriptive_statistics_xml(self, df):

        print("\nGenerating descriptive statistics XML...")

        stats = df.describe(include="all")

        root = ET.Element("DescriptiveStatistics")

        for column in stats.columns:

            col_node = ET.SubElement(root,"Column")
            ET.SubElement(col_node,"Name").text = column

            for stat_name, value in stats[column].items():

                stat_node = ET.SubElement(col_node,"Stat")
                ET.SubElement(stat_node,"Metric").text = str(stat_name)
                ET.SubElement(stat_node,"Value").text = str(value)

        tree = ET.ElementTree(root)

        output_path = os.path.join(self.xml_path,"Stage1","descriptive_statistics.xml")
        tree.write(output_path)

        print("Descriptive statistics XML saved:", output_path)

    # --------------------------------------------------
    # Register figure
    # --------------------------------------------------

    def register_figure(self, stage, figure_name):

        self.figure_counter[stage] += 1

        figure_number = f"Fig_{stage}_{self.figure_counter[stage]}"

        record = f"{figure_number} : {figure_name}"

        report_file = os.path.join(self.report_path,"figure_registry.txt")

        with open(report_file,"a",encoding="utf-8") as f:
            f.write(record+"\n")

        return figure_number

    # --------------------------------------------------
    # Save figure helper
    # --------------------------------------------------

    def save_figure(self, stage, figure_name, plt):

        figure_number = self.register_figure(stage, figure_name)

        file_name = f"{figure_number}_{figure_name}.png"

        save_path = os.path.join(self.diagrams_path,stage,file_name)

        plt.savefig(save_path)

        print("Figure saved:", save_path)

    # --------------------------------------------------
    # Generate XML summary
    # --------------------------------------------------

    def generate_xml_summary(self):

        print("\nGenerating XML summary report...")

        root = ET.Element("XMLSummary")

        for stage in self.stages:

            stage_node = ET.SubElement(root,"Stage")

            ET.SubElement(stage_node,"Name").text = stage

            stage_path = os.path.join(self.xml_path,stage)

            files = os.listdir(stage_path)

            for file in files:

                file_node = ET.SubElement(stage_node,"File")
                file_node.text = file

        tree = ET.ElementTree(root)

        output_path = os.path.join(self.report_path,"xml_summary.xml")

        tree.write(output_path)

        print("XML summary generated:", output_path)

    # --------------------------------------------------
    # Run initialization
    # --------------------------------------------------

    def initialize_research_environment(self):

        print("\nInitializing research environment...")

        self.create_directories()

        df = self.load_dataset()

        self.generate_dataset_profile_xml(df)

        self.generate_descriptive_statistics_xml(df)

        self.generate_xml_summary()

        print("\nResearch environment ready.")

        return df


# --------------------------------------------------
# Execution
# --------------------------------------------------

def run():

    manager = ResearchArtifactManager()

    df = manager.initialize_research_environment()

    return manager, df


# Run automatically in Jupyter

manager, df = run()


=== Research Artifact Manager Initialized ===

Initializing research environment...

Creating folder structure...
Folder structure created successfully.

Loading dataset...
Dataset loaded successfully.
Rows: 1069
Columns: 15

Generating dataset profile XML...
Dataset profile XML saved: D:\Med_data\XML\Stage1\dataset_profile.xml

Generating descriptive statistics XML...
Descriptive statistics XML saved: D:\Med_data\XML\Stage1\descriptive_statistics.xml

Generating XML summary report...
XML summary generated: D:\Med_data\Reports\xml_summary.xml

Research environment ready.


In [5]:
import pandas as pd

df = pd.read_csv(r"d:\med_data\healthcare_dataset.csv")
print(df.shape)

(1069, 15)


In [1]:
import pandas as pd

dataset_path = r"D:\Med_data\healthcare_dataset.csv"

df = pd.read_csv(dataset_path)

print(df.shape)
print(df.head())
print(df.columns)

(1069, 15)
            Name  Age  Gender Blood Type Medical Condition Date of Admission  \
0  Bobby JacksOn   30    Male         B-            Cancer         1/31/2024   
1   LesLie TErRy   62    Male         A+           Obesity         8/20/2019   
2    DaNnY sMitH   76  Female         A-           Obesity         9/22/2022   
3   andrEw waTtS   28  Female         O+          Diabetes        11/18/2020   
4  adrIENNE bEll   43  Female        AB+            Cancer         9/19/2022   

             Doctor                    Hospital Insurance Provider  \
0     Matthew Smith             Sons and Miller         Blue Cross   
1   Samantha Davies                     Kim Inc           Medicare   
2  Tiffany Mitchell                    Cook PLC              Aetna   
3       Kevin Wells  Hernandez Rogers and Vang,           Medicare   
4    Kathleen Hanna                 White-White              Aetna   

   Billing Amount  Room Number Admission Type Discharge Date   Medication  \
0     1885

In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

print("\n==============================")
print("Stage 1 : Healthcare Data Exploration (Stable Version)")
print("==============================")

# -----------------------------
# Dataset path
# -----------------------------

dataset_path = r"D:\Med_data\healthcare_dataset.csv"

# -----------------------------
# Output folders
# -----------------------------

diagram_path = r"D:\Med_data\Diagrams\Stage1"
xml_path = r"D:\Med_data\XML\Stage1"

os.makedirs(diagram_path, exist_ok=True)
os.makedirs(xml_path, exist_ok=True)

# -----------------------------
# Load dataset safely
# -----------------------------

df = pd.read_csv(dataset_path, encoding="latin1")

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# -----------------------------
# Safe date conversion
# -----------------------------

df["Date of Admission"] = pd.to_datetime(df["Date of Admission"], errors="coerce")
df["Discharge Date"] = pd.to_datetime(df["Discharge Date"], errors="coerce")

# -----------------------------
# Derived feature
# -----------------------------

df["Length_of_Stay"] = (df["Discharge Date"] - df["Date of Admission"]).dt.days

# Replace invalid values
df["Length_of_Stay"] = df["Length_of_Stay"].fillna(0)

# -----------------------------
# Figure counter
# -----------------------------

figure_counter = 0

def save_plot(name):

    global figure_counter
    figure_counter += 1

    filename = f"Fig_Stage1_{figure_counter}_{name}.png"
    path = os.path.join(diagram_path, filename)

    plt.savefig(path)
    plt.close()

    print("Saved:", filename)

# -----------------------------
# Age distribution
# -----------------------------

plt.figure()
plt.hist(df["Age"], bins=20)
plt.title("Age Distribution")
save_plot("Age_distribution")

# -----------------------------
# Billing distribution
# -----------------------------

plt.figure()
plt.hist(df["Billing Amount"], bins=20)
plt.title("Billing Amount Distribution")
save_plot("Billing_distribution")

# -----------------------------
# Length of stay distribution
# -----------------------------

plt.figure()
plt.hist(df["Length_of_Stay"], bins=20)
plt.title("Length of Stay Distribution")
save_plot("Length_of_Stay_distribution")

# -----------------------------
# Gender distribution
# -----------------------------

plt.figure()
df["Gender"].value_counts().plot(kind="bar")
plt.title("Gender Distribution")
save_plot("Gender_distribution")

# -----------------------------
# Blood type distribution
# -----------------------------

plt.figure()
df["Blood Type"].value_counts().plot(kind="bar")
plt.title("Blood Type Distribution")
save_plot("BloodType_distribution")

# -----------------------------
# Admission type distribution
# -----------------------------

plt.figure()
df["Admission Type"].value_counts().plot(kind="bar")
plt.title("Admission Type Distribution")
save_plot("AdmissionType_distribution")

# -----------------------------
# Medical condition (Top 10 only)
# -----------------------------

plt.figure()
df["Medical Condition"].value_counts().head(10).plot(kind="bar")
plt.title("Top Medical Conditions")
save_plot("MedicalCondition_distribution")

# -----------------------------
# Medication distribution (Top 10)
# -----------------------------

plt.figure()
df["Medication"].value_counts().head(10).plot(kind="bar")
plt.title("Top Medications")
save_plot("Medication_distribution")

# -----------------------------
# Test result distribution
# -----------------------------

plt.figure()
df["Test Results"].value_counts().plot(kind="bar")
plt.title("Test Results Distribution")
save_plot("TestResults_distribution")

# -----------------------------
# Age vs Billing scatter
# -----------------------------

plt.figure()
plt.scatter(df["Age"], df["Billing Amount"])
plt.title("Age vs Billing Amount")
save_plot("Age_vs_Billing")

# -----------------------------
# Correlation heatmap (manual)
# -----------------------------

numeric = df[["Age","Billing Amount","Length_of_Stay"]]
corr = numeric.corr()

plt.figure()
plt.imshow(corr)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar()
plt.title("Correlation Matrix")
save_plot("Correlation_heatmap")

# -----------------------------
# XML report : distribution summary
# -----------------------------

root = ET.Element("DistributionAnalysis")

columns = [
"Gender",
"Blood Type",
"Admission Type",
"Medical Condition",
"Medication",
"Test Results"
]

for column in columns:

    col_node = ET.SubElement(root,"Column")
    ET.SubElement(col_node,"Name").text = column

    counts = df[column].value_counts()

    for value,count in counts.items():

        val_node = ET.SubElement(col_node,"Value")
        ET.SubElement(val_node,"Category").text = str(value)
        ET.SubElement(val_node,"Count").text = str(count)

tree = ET.ElementTree(root)

xml_file = os.path.join(xml_path,"distribution_analysis.xml")
tree.write(xml_file)

print("Saved XML:", xml_file)

# -----------------------------
# XML report : stay statistics
# -----------------------------

root = ET.Element("StayStatistics")

ET.SubElement(root,"MeanStay").text = str(df["Length_of_Stay"].mean())
ET.SubElement(root,"MedianStay").text = str(df["Length_of_Stay"].median())
ET.SubElement(root,"MaxStay").text = str(df["Length_of_Stay"].max())
ET.SubElement(root,"MinStay").text = str(df["Length_of_Stay"].min())

tree = ET.ElementTree(root)

xml_file = os.path.join(xml_path,"stay_statistics.xml")
tree.write(xml_file)

print("Saved XML:", xml_file)

print("\nStage 1 Completed Successfully")


Stage 1 : Healthcare Data Exploration (Stable Version)
Dataset loaded successfully
Rows: 1069
Columns: 15
Saved: Fig_Stage1_1_Age_distribution.png
Saved: Fig_Stage1_2_Billing_distribution.png
Saved: Fig_Stage1_3_Length_of_Stay_distribution.png
Saved: Fig_Stage1_4_Gender_distribution.png
Saved: Fig_Stage1_5_BloodType_distribution.png
Saved: Fig_Stage1_6_AdmissionType_distribution.png
Saved: Fig_Stage1_7_MedicalCondition_distribution.png
Saved: Fig_Stage1_8_Medication_distribution.png
Saved: Fig_Stage1_9_TestResults_distribution.png
Saved: Fig_Stage1_10_Age_vs_Billing.png
Saved: Fig_Stage1_11_Correlation_heatmap.png
Saved XML: D:\Med_data\XML\Stage1\distribution_analysis.xml
Saved XML: D:\Med_data\XML\Stage1\stay_statistics.xml

Stage 1 Completed Successfully


In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

print("\n==============================")
print("Stage 2 : Data Preprocessing")
print("==============================")

# --------------------------------
# Dataset path
# --------------------------------

dataset_path = r"D:\Med_data\healthcare_dataset.csv"

# --------------------------------
# Output folders
# --------------------------------

diagram_path = r"D:\Med_data\Diagrams\Stage2"
xml_path = r"D:\Med_data\XML\Stage2"

os.makedirs(diagram_path, exist_ok=True)
os.makedirs(xml_path, exist_ok=True)

# --------------------------------
# Load dataset
# --------------------------------

df = pd.read_csv(dataset_path, encoding="latin1")

print("Dataset loaded")

# --------------------------------
# Safe date conversion
# --------------------------------

df["Date of Admission"] = pd.to_datetime(df["Date of Admission"], errors="coerce")
df["Discharge Date"] = pd.to_datetime(df["Discharge Date"], errors="coerce")

df["Length_of_Stay"] = (df["Discharge Date"] - df["Date of Admission"]).dt.days
df["Length_of_Stay"] = df["Length_of_Stay"].fillna(0)

# --------------------------------
# Handle missing values
# --------------------------------

missing_before = df.isnull().sum()

df = df.fillna("Unknown")

missing_after = df.isnull().sum()

print("Missing values handled")

# --------------------------------
# Encoding categorical variables
# --------------------------------

categorical_columns = [
"Gender",
"Blood Type",
"Medical Condition",
"Admission Type",
"Medication",
"Test Results"
]

encoding_maps = {}

for col in categorical_columns:

    df[col] = df[col].astype(str)

    unique_vals = df[col].unique()

    mapping = {val:i for i,val in enumerate(unique_vals)}

    encoding_maps[col] = mapping

    df[col] = df[col].map(mapping)

print("Categorical variables encoded")

# --------------------------------
# Save encoded dataset
# --------------------------------

encoded_dataset_path = r"D:\Med_data\processed_dataset.csv"

df.to_csv(encoded_dataset_path, index=False)

print("Processed dataset saved")

# --------------------------------
# Figure counter
# --------------------------------

figure_counter = 0

def save_plot(name):

    global figure_counter
    figure_counter += 1

    filename = f"Fig_Stage2_{figure_counter}_{name}.png"
    path = os.path.join(diagram_path, filename)

    plt.savefig(path)
    plt.close()

    print("Saved:", filename)

# --------------------------------
# Length of stay distribution
# --------------------------------

plt.figure()
plt.hist(df["Length_of_Stay"], bins=20)
plt.title("Length of Stay Distribution")
save_plot("Length_of_Stay_distribution")

# --------------------------------
# Billing normalization check
# --------------------------------

plt.figure()
plt.hist(df["Billing Amount"], bins=20)
plt.title("Billing Amount Distribution After Cleaning")
save_plot("Billing_distribution")

# --------------------------------
# Encoded gender distribution
# --------------------------------

plt.figure()
df["Gender"].value_counts().plot(kind="bar")
plt.title("Encoded Gender Distribution")
save_plot("Gender_encoded_distribution")

# --------------------------------
# Encoded admission type
# --------------------------------

plt.figure()
df["Admission Type"].value_counts().plot(kind="bar")
plt.title("Encoded Admission Type Distribution")
save_plot("AdmissionType_encoded_distribution")

# --------------------------------
# Test result encoded
# --------------------------------

plt.figure()
df["Test Results"].value_counts().plot(kind="bar")
plt.title("Encoded Test Results Distribution")
save_plot("TestResults_encoded_distribution")

# --------------------------------
# XML report : preprocessing summary
# --------------------------------

root = ET.Element("PreprocessingReport")

missing_node = ET.SubElement(root,"MissingValuesBefore")

for col,val in missing_before.items():

    col_node = ET.SubElement(missing_node,"Column")
    ET.SubElement(col_node,"Name").text = col
    ET.SubElement(col_node,"Missing").text = str(val)

missing_after_node = ET.SubElement(root,"MissingValuesAfter")

for col,val in missing_after.items():

    col_node = ET.SubElement(missing_after_node,"Column")
    ET.SubElement(col_node,"Name").text = col
    ET.SubElement(col_node,"Missing").text = str(val)

encoding_node = ET.SubElement(root,"EncodingMaps")

for col,map_dict in encoding_maps.items():

    col_node = ET.SubElement(encoding_node,"Column")
    ET.SubElement(col_node,"Name").text = col

    for key,val in map_dict.items():

        map_node = ET.SubElement(col_node,"Mapping")
        ET.SubElement(map_node,"Original").text = str(key)
        ET.SubElement(map_node,"Encoded").text = str(val)

tree = ET.ElementTree(root)

xml_file = os.path.join(xml_path,"preprocessing_report.xml")

tree.write(xml_file)

print("Saved XML:", xml_file)

print("\nStage 2 Completed Successfully")


Stage 2 : Data Preprocessing
Dataset loaded
Missing values handled
Categorical variables encoded
Processed dataset saved
Saved: Fig_Stage2_1_Length_of_Stay_distribution.png
Saved: Fig_Stage2_2_Billing_distribution.png
Saved: Fig_Stage2_3_Gender_encoded_distribution.png
Saved: Fig_Stage2_4_AdmissionType_encoded_distribution.png
Saved: Fig_Stage2_5_TestResults_encoded_distribution.png
Saved XML: D:\Med_data\XML\Stage2\preprocessing_report.xml

Stage 2 Completed Successfully


In [7]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

print("\n==============================")
print("Stage 3 : Statistical Analysis")
print("==============================")

# --------------------------------
# Dataset path (output of Stage 2)
# --------------------------------

dataset_path = r"D:\Med_data\processed_dataset.csv"

# --------------------------------
# Output folders
# --------------------------------

diagram_path = r"D:\Med_data\Diagrams\Stage3"
xml_path = r"D:\Med_data\XML\Stage3"

os.makedirs(diagram_path, exist_ok=True)
os.makedirs(xml_path, exist_ok=True)

# --------------------------------
# Load dataset
# --------------------------------

df = pd.read_csv(dataset_path)

print("Processed dataset loaded")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# --------------------------------
# Figure counter
# --------------------------------

figure_counter = 0

def save_plot(name):

    global figure_counter
    figure_counter += 1

    filename = f"Fig_Stage3_{figure_counter}_{name}.png"
    path = os.path.join(diagram_path, filename)

    plt.savefig(path)
    plt.close()

    print("Saved:", filename)

# --------------------------------
# Descriptive statistics
# --------------------------------

stats = df.describe()

print("\nDescriptive Statistics:")
print(stats)

# --------------------------------
# Age distribution (refined)
# --------------------------------

plt.figure()
plt.hist(df["Age"], bins=25)
plt.title("Age Distribution (Detailed)")
save_plot("Age_distribution_detailed")

# --------------------------------
# Billing amount distribution
# --------------------------------

plt.figure()
plt.hist(df["Billing Amount"], bins=25)
plt.title("Billing Amount Distribution")
save_plot("Billing_amount_distribution")

# --------------------------------
# Length of stay distribution
# --------------------------------

plt.figure()
plt.hist(df["Length_of_Stay"], bins=25)
plt.title("Length of Stay Distribution")
save_plot("Stay_length_distribution")

# --------------------------------
# Medical cost insights
# --------------------------------

cost_by_condition = df.groupby("Medical Condition")["Billing Amount"].mean()

plt.figure()
cost_by_condition.sort_values(ascending=False).head(10).plot(kind="bar")
plt.title("Average Billing by Medical Condition (Top 10)")
save_plot("Cost_by_condition")

# --------------------------------
# Admission type vs billing
# --------------------------------

cost_by_admission = df.groupby("Admission Type")["Billing Amount"].mean()

plt.figure()
cost_by_admission.plot(kind="bar")
plt.title("Average Billing by Admission Type")
save_plot("Cost_by_admission_type")

# --------------------------------
# Correlation matrix
# --------------------------------

numeric_cols = ["Age","Billing Amount","Length_of_Stay"]

corr = df[numeric_cols].corr()

plt.figure()
plt.imshow(corr)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar()
plt.title("Correlation Matrix")
save_plot("Correlation_matrix")

# --------------------------------
# Age vs length of stay
# --------------------------------

plt.figure()
plt.scatter(df["Age"], df["Length_of_Stay"])
plt.title("Age vs Length of Stay")
save_plot("Age_vs_stay")

# --------------------------------
# Billing vs stay
# --------------------------------

plt.figure()
plt.scatter(df["Billing Amount"], df["Length_of_Stay"])
plt.title("Billing Amount vs Length of Stay")
save_plot("Billing_vs_stay")

# --------------------------------
# XML report : descriptive statistics
# --------------------------------

root = ET.Element("DescriptiveStatistics")

for column in stats.columns:

    col_node = ET.SubElement(root,"Column")
    ET.SubElement(col_node,"Name").text = column

    for stat,value in stats[column].items():

        stat_node = ET.SubElement(col_node,"Statistic")
        ET.SubElement(stat_node,"Metric").text = stat
        ET.SubElement(stat_node,"Value").text = str(value)

tree = ET.ElementTree(root)

xml_file = os.path.join(xml_path,"descriptive_statistics.xml")
tree.write(xml_file)

print("Saved XML:", xml_file)

# --------------------------------
# XML report : correlation results
# --------------------------------

root = ET.Element("CorrelationResults")

for i in corr.index:

    row_node = ET.SubElement(root,"Variable")
    ET.SubElement(row_node,"Name").text = i

    for j in corr.columns:

        val = corr.loc[i,j]

        cell = ET.SubElement(row_node,"Correlation")
        ET.SubElement(cell,"With").text = j
        ET.SubElement(cell,"Value").text = str(val)

tree = ET.ElementTree(root)

xml_file = os.path.join(xml_path,"correlation_results.xml")
tree.write(xml_file)

print("Saved XML:", xml_file)

print("\nStage 3 Completed Successfully")


Stage 3 : Statistical Analysis
Processed dataset loaded
Rows: 1069
Columns: 16

Descriptive Statistics:
               Age       Gender   Blood Type  Medical Condition  \
count  1069.000000  1069.000000  1069.000000        1069.000000   
mean     51.049579     0.475210     3.446211           2.538821   
std      19.629489     0.499619     2.266049           1.724368   
min      18.000000     0.000000     0.000000           0.000000   
25%      34.000000     0.000000     1.000000           1.000000   
50%      51.000000     0.000000     3.000000           3.000000   
75%      67.000000     1.000000     5.000000           4.000000   
max      85.000000     1.000000     7.000000           5.000000   

       Billing Amount  Room Number  Admission Type   Medication  Test Results  \
count     1069.000000  1069.000000     1069.000000  1069.000000   1069.000000   
mean     25002.387010   294.461179        1.051450     1.980355      1.039289   
std      14466.047356   116.602070        0.8223

In [9]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

print("\n==============================")
print("Stage 4 : Machine Learning Prediction")
print("==============================")

# --------------------------------
# Dataset path
# --------------------------------

dataset_path = r"D:\Med_data\processed_dataset.csv"

# --------------------------------
# Output folders
# --------------------------------

diagram_path = r"D:\Med_data\Diagrams\Stage4"
xml_path = r"D:\Med_data\XML\Stage4"

os.makedirs(diagram_path, exist_ok=True)
os.makedirs(xml_path, exist_ok=True)

# --------------------------------
# Load dataset
# --------------------------------

df = pd.read_csv(dataset_path)

print("Processed dataset loaded")
print("Rows:", df.shape[0])

# --------------------------------
# Target variable
# --------------------------------

target_column = "Test Results"

# --------------------------------
# Features
# --------------------------------

feature_columns = [
"Age",
"Billing Amount",
"Length_of_Stay",
"Gender",
"Blood Type",
"Medical Condition",
"Admission Type",
"Medication"
]

X = df[feature_columns]
y = df[target_column]

# --------------------------------
# Train Test Split
# --------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape[0])
print("Testing size:", X_test.shape[0])

# --------------------------------
# Models
# --------------------------------

models = {
"DecisionTree": DecisionTreeClassifier(),
"RandomForest": RandomForestClassifier(),
"LogisticRegression": LogisticRegression(max_iter=200),
"KNN": KNeighborsClassifier()
}

results = {}

# --------------------------------
# Figure counter
# --------------------------------

figure_counter = 0

def save_plot(name):

    global figure_counter
    figure_counter += 1

    filename = f"Fig_Stage4_{figure_counter}_{name}.png"
    path = os.path.join(diagram_path, filename)

    plt.savefig(path)
    plt.close()

    print("Saved:", filename)

# --------------------------------
# Train models
# --------------------------------

for model_name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    acc = accuracy_score(y_test, predictions)

    results[model_name] = acc

    print(model_name, "Accuracy:", acc)

    # Confusion matrix

    cm = confusion_matrix(y_test, predictions)

    plt.figure()
    plt.imshow(cm)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    save_plot(f"confusion_matrix_{model_name}")

# --------------------------------
# Accuracy comparison plot
# --------------------------------

plt.figure()

names = list(results.keys())
values = list(results.values())

plt.bar(names, values)

plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")

save_plot("model_accuracy_comparison")

# --------------------------------
# Feature importance (Random Forest)
# --------------------------------

rf_model = models["RandomForest"]

importance = rf_model.feature_importances_

plt.figure()

plt.bar(feature_columns, importance)
plt.xticks(rotation=45)

plt.title("Feature Importance - Random Forest")

save_plot("feature_importance_random_forest")

# --------------------------------
# XML report : ML experiment
# --------------------------------

root = ET.Element("MLExperimentResults")

for model_name, acc in results.items():

    model_node = ET.SubElement(root,"Model")

    ET.SubElement(model_node,"Name").text = model_name
    ET.SubElement(model_node,"Accuracy").text = str(acc)

tree = ET.ElementTree(root)

xml_file = os.path.join(xml_path,"ml_experiment_results.xml")

tree.write(xml_file)

print("Saved XML:", xml_file)

print("\nStage 4 Completed Successfully")


Stage 4 : Machine Learning Prediction
Processed dataset loaded
Rows: 1069
Training size: 855
Testing size: 214
DecisionTree Accuracy: 0.3037383177570093
Saved: Fig_Stage4_1_confusion_matrix_DecisionTree.png
RandomForest Accuracy: 0.308411214953271
Saved: Fig_Stage4_2_confusion_matrix_RandomForest.png


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression Accuracy: 0.3925233644859813
Saved: Fig_Stage4_3_confusion_matrix_LogisticRegression.png
KNN Accuracy: 0.2803738317757009
Saved: Fig_Stage4_4_confusion_matrix_KNN.png
Saved: Fig_Stage4_5_model_accuracy_comparison.png
Saved: Fig_Stage4_6_feature_importance_random_forest.png
Saved XML: D:\Med_data\XML\Stage4\ml_experiment_results.xml

Stage 4 Completed Successfully


In [11]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("\n==============================")
print("Stage 5 : Patient Segmentation")
print("==============================")

# --------------------------------
# Dataset path
# --------------------------------

dataset_path = r"D:\Med_data\processed_dataset.csv"

# --------------------------------
# Output folders
# --------------------------------

diagram_path = r"D:\Med_data\Diagrams\Stage5"
xml_path = r"D:\Med_data\XML\Stage5"

os.makedirs(diagram_path, exist_ok=True)
os.makedirs(xml_path, exist_ok=True)

# --------------------------------
# Load dataset
# --------------------------------

df = pd.read_csv(dataset_path)

print("Processed dataset loaded")
print("Rows:", df.shape[0])

# --------------------------------
# Features used for clustering
# --------------------------------

features = [
"Age",
"Billing Amount",
"Length_of_Stay",
"Gender",
"Blood Type",
"Medical Condition",
"Admission Type",
"Medication"
]

X = df[features]

# --------------------------------
# Standardize features
# --------------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --------------------------------
# KMeans clustering
# --------------------------------

k = 3
kmeans = KMeans(n_clusters=k, random_state=42)

clusters = kmeans.fit_predict(X_scaled)

df["Cluster"] = clusters

print("Clustering completed")

# --------------------------------
# PCA for visualization
# --------------------------------

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df["PCA1"] = X_pca[:,0]
df["PCA2"] = X_pca[:,1]

# --------------------------------
# Figure counter
# --------------------------------

figure_counter = 0

def save_plot(name):

    global figure_counter
    figure_counter += 1

    filename = f"Fig_Stage5_{figure_counter}_{name}.png"
    path = os.path.join(diagram_path, filename)

    plt.savefig(path)
    plt.close()

    print("Saved:", filename)

# --------------------------------
# Cluster scatter plot
# --------------------------------

plt.figure()

for cluster_id in range(k):

    cluster_data = df[df["Cluster"] == cluster_id]

    plt.scatter(
        cluster_data["PCA1"],
        cluster_data["PCA2"],
        label=f"Cluster {cluster_id}"
    )

plt.legend()
plt.title("Patient Clusters (PCA Projection)")

save_plot("cluster_scatter")

# --------------------------------
# Cluster distribution
# --------------------------------

plt.figure()

df["Cluster"].value_counts().sort_index().plot(kind="bar")

plt.title("Cluster Distribution")

save_plot("cluster_distribution")

# --------------------------------
# Cluster feature means
# --------------------------------

cluster_means = df.groupby("Cluster")[features].mean()

plt.figure()

plt.imshow(cluster_means)

plt.colorbar()

plt.xticks(range(len(features)), features, rotation=45)
plt.yticks(range(k), [f"Cluster {i}" for i in range(k)])

plt.title("Cluster Feature Heatmap")

save_plot("cluster_feature_heatmap")

# --------------------------------
# XML : cluster profiles
# --------------------------------

root = ET.Element("ClusterProfiles")

for cluster_id in range(k):

    cluster_node = ET.SubElement(root,"Cluster")

    ET.SubElement(cluster_node,"ClusterID").text = str(cluster_id)

    cluster_subset = df[df["Cluster"] == cluster_id]

    ET.SubElement(cluster_node,"Size").text = str(len(cluster_subset))

    means = cluster_subset[features].mean()

    features_node = ET.SubElement(cluster_node,"FeatureMeans")

    for f,val in means.items():

        feature_node = ET.SubElement(features_node,"Feature")

        ET.SubElement(feature_node,"Name").text = f
        ET.SubElement(feature_node,"Value").text = str(val)

tree = ET.ElementTree(root)

xml_file = os.path.join(xml_path,"cluster_profiles.xml")

tree.write(xml_file)

print("Saved XML:", xml_file)

print("\nStage 5 Completed Successfully")


Stage 5 : Patient Segmentation
Processed dataset loaded
Rows: 1069
Clustering completed
Saved: Fig_Stage5_1_cluster_scatter.png
Saved: Fig_Stage5_2_cluster_distribution.png
Saved: Fig_Stage5_3_cluster_feature_heatmap.png
Saved XML: D:\Med_data\XML\Stage5\cluster_profiles.xml

Stage 5 Completed Successfully


In [13]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

print("\n==============================")
print("Stage 6 : LLM Clinical Insight Engine")
print("==============================")

# --------------------------------
# Paths
# --------------------------------

cluster_xml = r"D:\Med_data\XML\Stage5\cluster_profiles.xml"
ml_xml = r"D:\Med_data\XML\Stage4\ml_experiment_results.xml"

diagram_path = r"D:\Med_data\Diagrams\Stage6"
xml_path = r"D:\Med_data\XML\Stage6"

os.makedirs(diagram_path, exist_ok=True)
os.makedirs(xml_path, exist_ok=True)

# --------------------------------
# Read ML experiment results
# --------------------------------

ml_tree = ET.parse(ml_xml)
ml_root = ml_tree.getroot()

ml_results = {}

for model in ml_root.findall("Model"):
    name = model.find("Name").text
    acc = float(model.find("Accuracy").text)
    ml_results[name] = acc

print("Loaded ML experiment results")

# --------------------------------
# Identify best model
# --------------------------------

best_model = max(ml_results, key=ml_results.get)
best_accuracy = ml_results[best_model]

print("Best Model:", best_model)
print("Accuracy:", best_accuracy)

# --------------------------------
# Read cluster profiles
# --------------------------------

cluster_tree = ET.parse(cluster_xml)
cluster_root = cluster_tree.getroot()

clusters = []

for cluster in cluster_root.findall("Cluster"):

    cid = cluster.find("ClusterID").text
    size = cluster.find("Size").text

    clusters.append({
        "ClusterID": cid,
        "Size": int(size)
    })

print("Loaded cluster profiles")

# --------------------------------
# Generate LLM-style explanations
# --------------------------------

cluster_explanations = []

for c in clusters:

    cid = c["ClusterID"]
    size = c["Size"]

    explanation = f"""
Cluster {cid} represents a group of {size} patients with similar healthcare characteristics.
This cluster may reflect a distinct clinical profile based on medical conditions,
treatment patterns, hospital stay duration, and healthcare cost indicators.

Such clusters can help healthcare administrators identify patient groups
requiring specialized care strategies or cost management interventions.
"""

    cluster_explanations.append({
        "ClusterID": cid,
        "Explanation": explanation
    })

# --------------------------------
# Healthcare insight summary
# --------------------------------

summary_text = f"""
The machine learning experiments identified {best_model} as the best predictive model
with an accuracy of {best_accuracy:.2f}. This model can be used to support predictive
healthcare analytics such as patient risk prediction and treatment outcome forecasting.

Cluster analysis revealed {len(clusters)} distinct patient groups, highlighting
heterogeneity in patient demographics, healthcare utilization, and medical conditions.

Combining machine learning predictions with cluster-based patient segmentation
enables explainable healthcare analytics and supports data-driven clinical decisions.
"""

print("\nGenerated clinical insight summary")

# --------------------------------
# Visualization : cluster sizes
# --------------------------------

cluster_ids = [c["ClusterID"] for c in clusters]
cluster_sizes = [c["Size"] for c in clusters]

plt.figure()

plt.bar(cluster_ids, cluster_sizes)

plt.title("Cluster Size Overview")
plt.xlabel("Cluster")
plt.ylabel("Number of Patients")

figure_name = "Fig_Stage6_1_cluster_size_overview.png"
plt.savefig(os.path.join(diagram_path, figure_name))
plt.close()

print("Saved:", figure_name)

# --------------------------------
# Visualization : model accuracy
# --------------------------------

plt.figure()

names = list(ml_results.keys())
values = list(ml_results.values())

plt.bar(names, values)

plt.title("ML Model Accuracy Overview")
plt.ylabel("Accuracy")

figure_name = "Fig_Stage6_2_model_accuracy_summary.png"
plt.savefig(os.path.join(diagram_path, figure_name))
plt.close()

print("Saved:", figure_name)

# --------------------------------
# XML report : LLM insights
# --------------------------------

root = ET.Element("LLMClinicalInsights")

summary_node = ET.SubElement(root, "Summary")
summary_node.text = summary_text

best_model_node = ET.SubElement(root, "BestModel")

ET.SubElement(best_model_node, "ModelName").text = best_model
ET.SubElement(best_model_node, "Accuracy").text = str(best_accuracy)

clusters_node = ET.SubElement(root, "ClusterInsights")

for exp in cluster_explanations:

    cluster_node = ET.SubElement(clusters_node, "Cluster")

    ET.SubElement(cluster_node, "ClusterID").text = exp["ClusterID"]
    ET.SubElement(cluster_node, "Explanation").text = exp["Explanation"]

tree = ET.ElementTree(root)

xml_file = os.path.join(xml_path, "clinical_insights.xml")

tree.write(xml_file)

print("Saved XML:", xml_file)

print("\nStage 6 Completed Successfully")


Stage 6 : LLM Clinical Insight Engine
Loaded ML experiment results
Best Model: LogisticRegression
Accuracy: 0.3925233644859813
Loaded cluster profiles

Generated clinical insight summary
Saved: Fig_Stage6_1_cluster_size_overview.png
Saved: Fig_Stage6_2_model_accuracy_summary.png
Saved XML: D:\Med_data\XML\Stage6\clinical_insights.xml

Stage 6 Completed Successfully


In [15]:
import os
import xml.etree.ElementTree as ET

print("\n==============================")
print("Thesis Artifact Generator")
print("==============================")

# --------------------------------
# Base paths
# --------------------------------

base_path = r"D:\Med_data"

diagram_root = os.path.join(base_path, "Diagrams")
xml_root = os.path.join(base_path, "XML")
report_root = os.path.join(base_path, "Reports")

os.makedirs(report_root, exist_ok=True)

# --------------------------------
# Stage folders
# --------------------------------

stages = [
"Stage1",
"Stage2",
"Stage3",
"Stage4",
"Stage5",
"Stage6"
]

# --------------------------------
# Collect figures
# --------------------------------

figure_list = []

for stage in stages:

    stage_path = os.path.join(diagram_root, stage)

    if os.path.exists(stage_path):

        files = os.listdir(stage_path)

        for f in files:

            if f.endswith(".png"):

                figure_list.append({
                    "Stage": stage,
                    "File": f
                })

print("Collected figures:", len(figure_list))

# --------------------------------
# Write List of Figures
# --------------------------------

list_file = os.path.join(report_root, "list_of_figures.txt")

with open(list_file, "w", encoding="utf-8") as f:

    for i,fig in enumerate(figure_list, start=1):

        line = f"Figure {i}: {fig['File']} (from {fig['Stage']})\n"
        f.write(line)

print("Saved:", list_file)

# --------------------------------
# Generate Figure Captions
# --------------------------------

caption_file = os.path.join(report_root, "figure_captions.txt")

with open(caption_file, "w", encoding="utf-8") as f:

    for i,fig in enumerate(figure_list, start=1):

        caption = f"""
Figure {i}: {fig['File']}

This figure was generated during {fig['Stage']} of the healthcare analytics pipeline.
It illustrates an important analytical result related to patient behavior,
healthcare cost patterns, predictive modeling, or cluster analysis.
"""

        f.write(caption)
        f.write("\n")

print("Saved:", caption_file)

# --------------------------------
# Thesis placement guide
# --------------------------------

placement_file = os.path.join(report_root, "thesis_placement_guide.txt")

with open(placement_file, "w", encoding="utf-8") as f:

    for i,fig in enumerate(figure_list, start=1):

        stage = fig["Stage"]

        if stage == "Stage1":
            chapter = "Chapter 4 (Dataset Exploration)"
        elif stage == "Stage2":
            chapter = "Chapter 4 (Data Preprocessing)"
        elif stage == "Stage3":
            chapter = "Chapter 4 (Statistical Analysis)"
        elif stage == "Stage4":
            chapter = "Chapter 4 (Machine Learning Results)"
        elif stage == "Stage5":
            chapter = "Chapter 4 (Patient Segmentation)"
        else:
            chapter = "Chapter 4 (LLM-Based Clinical Insights)"

        line = f"Figure {i} → {chapter}\n"

        f.write(line)

print("Saved:", placement_file)

# --------------------------------
# XML summary
# --------------------------------

xml_summary = os.path.join(report_root, "xml_summary.txt")

with open(xml_summary, "w", encoding="utf-8") as f:

    for stage in stages:

        stage_path = os.path.join(xml_root, stage)

        if os.path.exists(stage_path):

            files = os.listdir(stage_path)

            f.write(f"\n{stage} XML Files\n")

            for file in files:

                if file.endswith(".xml"):
                    f.write(file + "\n")

print("Saved:", xml_summary)

# --------------------------------
# Research summary
# --------------------------------

summary_file = os.path.join(report_root, "research_summary.txt")

summary_text = """
Healthcare Analytics Pipeline Summary

This research implemented a six-stage artificial intelligence pipeline
to analyze healthcare data and extract meaningful clinical insights.

Stage 1 – Dataset Exploration
Initial statistical exploration of the healthcare dataset including
patient demographics, billing patterns, and medical conditions.

Stage 2 – Data Preprocessing
Data cleaning, missing value handling, categorical encoding,
and preparation of machine learning datasets.

Stage 3 – Statistical Analysis
Descriptive statistics, cost analysis, correlation analysis,
and healthcare pattern identification.

Stage 4 – Machine Learning Prediction
Development and evaluation of predictive models including
Decision Tree, Random Forest, Logistic Regression, and KNN.

Stage 5 – Patient Segmentation
Unsupervised clustering used to identify hidden patient groups
based on healthcare utilization patterns.

Stage 6 – LLM-Based Clinical Insights
Integration of artificial intelligence techniques to generate
interpretable clinical insights and summaries of analytical results.

The pipeline automatically generated visualizations, XML reports,
and structured research artifacts supporting reproducible
healthcare data analysis.

"""

with open(summary_file, "w", encoding="utf-8") as f:
    f.write(summary_text)

print("Saved:", summary_file)

print("\nThesis Artifact Generation Completed Successfully")


Thesis Artifact Generator
Collected figures: 35
Saved: D:\Med_data\Reports\list_of_figures.txt
Saved: D:\Med_data\Reports\figure_captions.txt
Saved: D:\Med_data\Reports\thesis_placement_guide.txt
Saved: D:\Med_data\Reports\xml_summary.txt
Saved: D:\Med_data\Reports\research_summary.txt

Thesis Artifact Generation Completed Successfully


In [17]:
import os
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt

print("\n==============================")
print("Healthcare AI Pipeline Overview")
print("==============================")

# ----------------------------------------------------
# Paths
# ----------------------------------------------------

base_path = r"D:\Med_data"

diagram_root = os.path.join(base_path,"Diagrams")
xml_root = os.path.join(base_path,"XML")
report_root = os.path.join(base_path,"Reports")

os.makedirs(report_root, exist_ok=True)

stages = [
"Stage1",
"Stage2",
"Stage3",
"Stage4",
"Stage5",
"Stage6"
]

# ----------------------------------------------------
# Count artifacts
# ----------------------------------------------------

artifact_summary = []

total_figures = 0
total_xml = 0

for stage in stages:

    fig_path = os.path.join(diagram_root, stage)
    xml_path = os.path.join(xml_root, stage)

    fig_count = 0
    xml_count = 0

    if os.path.exists(fig_path):
        fig_count = len([f for f in os.listdir(fig_path) if f.endswith(".png")])

    if os.path.exists(xml_path):
        xml_count = len([f for f in os.listdir(xml_path) if f.endswith(".xml")])

    total_figures += fig_count
    total_xml += xml_count

    artifact_summary.append({
        "stage":stage,
        "figures":fig_count,
        "xml":xml_count
    })

print("Total Figures:", total_figures)
print("Total XML files:", total_xml)

# ----------------------------------------------------
# Stage descriptions
# ----------------------------------------------------

stage_descriptions = {

"Stage1":
"Healthcare dataset exploration including patient demographics, billing patterns, and medical condition distributions.",

"Stage2":
"Data preprocessing including missing value handling, categorical encoding, and generation of machine-learning ready datasets.",

"Stage3":
"Statistical analysis including descriptive statistics, healthcare cost analysis, correlation analysis, and patient stay patterns.",

"Stage4":
"Machine learning prediction models including Decision Tree, Random Forest, Logistic Regression, and KNN for predicting medical test outcomes.",

"Stage5":
"Patient segmentation using clustering techniques to identify hidden healthcare utilization patterns and patient groups.",

"Stage6":
"LLM-based clinical interpretation where analytical results are converted into explainable healthcare insights."
}

# ----------------------------------------------------
# Create research summary text
# ----------------------------------------------------

summary_text = f"""
Healthcare AI Research Pipeline Summary

This system implements a six-stage artificial intelligence pipeline for
healthcare analytics and clinical insight generation.

Total Figures Generated: {total_figures}
Total XML Reports Generated: {total_xml}

Stages Overview
"""

for stage in stages:

    desc = stage_descriptions[stage]

    for item in artifact_summary:
        if item["stage"] == stage:

            summary_text += f"""

{stage}
Description: {desc}
Figures Produced: {item['figures']}
XML Reports Produced: {item['xml']}
"""

summary_text += """

Overall Contribution

The developed pipeline integrates statistical analysis,
machine learning models, patient segmentation, and
LLM-based clinical interpretation into a unified
healthcare analytics framework.

This automated system generates research artifacts
including figures, XML reports, and structured
documentation supporting reproducible healthcare research.

"""

# ----------------------------------------------------
# Save summary file
# ----------------------------------------------------

summary_file = os.path.join(report_root,"pipeline_overview.txt")

with open(summary_file,"w",encoding="utf-8") as f:
    f.write(summary_text)

print("Saved:", summary_file)

# ----------------------------------------------------
# XML pipeline overview
# ----------------------------------------------------

root = ET.Element("HealthcarePipeline")

for stage in stages:

    stage_node = ET.SubElement(root,"Stage")

    ET.SubElement(stage_node,"Name").text = stage
    ET.SubElement(stage_node,"Description").text = stage_descriptions[stage]

    for item in artifact_summary:
        if item["stage"] == stage:

            ET.SubElement(stage_node,"Figures").text = str(item["figures"])
            ET.SubElement(stage_node,"XMLReports").text = str(item["xml"])

tree = ET.ElementTree(root)

xml_file = os.path.join(report_root,"pipeline_overview.xml")
tree.write(xml_file)

print("Saved:", xml_file)

# ----------------------------------------------------
# Pipeline architecture diagram
# ----------------------------------------------------

stage_labels = [
"Stage1\nExploration",
"Stage2\nPreprocessing",
"Stage3\nStatistics",
"Stage4\nML Models",
"Stage5\nSegmentation",
"Stage6\nLLM Insights"
]

x = range(len(stage_labels))
y = [1]*len(stage_labels)

plt.figure(figsize=(10,3))
plt.scatter(x,y)

for i,label in enumerate(stage_labels):
    plt.text(i,1,label,ha='center')

plt.plot(x,y)

plt.title("Healthcare AI Pipeline Architecture")

plt.axis("off")

diagram_file = os.path.join(report_root,"pipeline_summary_diagram.png")

plt.savefig(diagram_file)
plt.close()

print("Saved:", diagram_file)

print("\nPipeline Overview Generation Completed Successfully")


Healthcare AI Pipeline Overview
Total Figures: 35
Total XML files: 10
Saved: D:\Med_data\Reports\pipeline_overview.txt
Saved: D:\Med_data\Reports\pipeline_overview.xml
Saved: D:\Med_data\Reports\pipeline_summary_diagram.png

Pipeline Overview Generation Completed Successfully
